# Tutorial: Plan 004.5 Tag and Content Predicates

**Audience:** Python users who know Archiver's catalog basics and want to select current files by byte identity or active tags.

**Prerequisites:** an Archiver development environment and familiarity with `Catalog.files(root)`.

**Learning goals:** build immutable content/tag predicates, retrieve every current path for one content identity, combine predicate families, use provenance, and distinguish display bounds from complete reductions.

Every example uses disposable synthetic files. Query operations remain read-only; setup changes only catalog tag metadata.


## Outline

1. Create a small tagged catalog.
2. Build immutable predicate values.
3. Retrieve every path for one content identity.
4. Filter by active tags and provenance.
5. Compose filters and calculate complete reductions.
6. Observe lazy tag-state changes.
7. Complete an exercise and clean up.


In [ ]:
from __future__ import annotations

from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory

from archiver import Catalog, TagProvenance
from archiver.query import content, path, tags

temporary_directory = TemporaryDirectory(prefix="archiver-predicates-")
workspace = Path(temporary_directory.name)
root = workspace / "source"
root.mkdir()

sample_files = {
    "photos/original.jpg": b"same-photo",
    "photos/copy.jpg": b"same-photo",
    "notes.txt": b"notes",
    "draft.txt": b"draft",
    "empty/a.bin": b"",
    "empty/b.bin": b"",
    "raw.bin": b"raw",
}
for relative_path, data in sample_files.items():
    destination = root / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)

source_before = {path: path.read_bytes() for path in root.rglob("*") if path.is_file()}
catalog = Catalog.create(workspace / "catalog.sqlite")
scan_summary = catalog.scan_directory(root)

photo_id = catalog.content_for_path(root, PurePosixPath("photos/original.jpg"))
notes_id = catalog.content_for_path(root, PurePosixPath("notes.txt"))
draft_id = catalog.content_for_path(root, PurePosixPath("draft.txt"))
empty_id = catalog.content_for_path(root, PurePosixPath("empty/a.bin"))
user = TagProvenance("user", "tutorial", "1")
system = TagProvenance("system", "classifier", "1")
second_system = TagProvenance("system", "second-classifier", "1")

catalog.add_content_tag(photo_id, "photo", user)
catalog.add_content_tag(photo_id, "photo", system)
catalog.add_content_tag(photo_id, "photo", second_system)
catalog.add_content_tag(photo_id, "reviewed", user)
catalog.add_content_tag(notes_id, "document", system)
catalog.add_content_tag(notes_id, "reviewed", user)
catalog.add_content_tag(draft_id, "document", system)
catalog.add_content_tag(draft_id, "draft", user)
catalog.add_content_tag(empty_id, "empty", system)

assert scan_summary.files_observed == 7
scan_summary

## 1. Predicate factories create immutable values

`tags.all_of()` and `tags.any_of()` validate names, remove repeats, and sort them into canonical order. Like queries, predicate values can be safely reused.

In [ ]:

reviewed_photo = tags.all_of("reviewed", "photo", "reviewed")
draft_or_reviewed = tags.any_of("reviewed", "draft", "draft")

assert reviewed_photo.names == ("photo", "reviewed")
assert draft_or_reviewed.names == ("draft", "reviewed")
{
    "all_tags": reviewed_photo.names,
    "any_tags": draft_or_reviewed.names,
}

## 2. Retrieve every path for one content identity

Content identity describes bytes, not a pathname. Both JPEG paths therefore match the same predicate. `fetch()` remains bounded, while the deliberately explicit `stream_all()` terminal yields every matching current observation. This is the supported way to expand a selected duplicate group completely.


In [ ]:
# `content.is_()` records one complete `ContentId`. 
exact_photo = content.is_(photo_id)
exact_photo

In [ ]:
photo_query = catalog.files(root).where(exact_photo).limit(1)
preview_paths = [row.relative_path.as_posix() for row in photo_query.fetch()]
all_photo_paths = [row.relative_path.as_posix() for row in photo_query.stream_all()]

assert preview_paths == ["photos/copy.jpg"]
assert all_photo_paths == ["photos/copy.jpg", "photos/original.jpg"]
assert photo_query.count() == 2
assert photo_query.distinct_content_count() == 1
{"preview": preview_paths, "all_paths": all_photo_paths}

## 3. Filter by active tags and provenance

`all_of()` requires every name; `any_of()` requires at least one. The optional provenance restricts every requested tag to user or system assertions. Tags belong to content, so both photo paths match even though the assertions were added once. Multiple active producers of `photo` do not multiply those paths.


In [ ]:
# Reminder of the tagging:
# catalog.add_content_tag(photo_id, "photo", user)
# catalog.add_content_tag(photo_id, "photo", system)
# catalog.add_content_tag(photo_id, "photo", second_system)
# catalog.add_content_tag(photo_id, "reviewed", user)

# catalog.add_content_tag(notes_id, "document", system)
# catalog.add_content_tag(notes_id, "reviewed", user)

# catalog.add_content_tag(draft_id, "document", system)
# catalog.add_content_tag(draft_id, "draft", user)

# catalog.add_content_tag(empty_id, "empty", system)


In [ ]:
def paths_for(query):
    return [row.relative_path.as_posix() for row in query.fetch()]


reviewed_photos = catalog.files(root).where(tags.all_of("reviewed", "photo"))
system_documents = catalog.files(root).where(tags.all_of("document", provenance="system"))
user_documents = catalog.files(root).where(tags.all_of("document", provenance="user"))

assert paths_for(reviewed_photos) == ["photos/copy.jpg", "photos/original.jpg"]
assert paths_for(system_documents) == ["draft.txt", "notes.txt"]
assert paths_for(user_documents) == []
{
    "reviewed photos": paths_for(reviewed_photos),
    "system documents": paths_for(system_documents),
    "user documents": paths_for(user_documents),
}

## 4. Compose predicates and ask complete questions

Repeated `where()` calls mean logical AND, even across predicate families. Here the tag predicate first finds reviewed or draft content and the path predicate retains only text files. The one-row display limit affects `fetch()` but not `count()`, `distinct_content_count()`, or `total_size_bytes()`.


In [ ]:
# unlimited fetch should match 2 files here:
text_work = catalog.files(root).where(tags.any_of("reviewed", "draft")).where(path.glob("*.txt")).limit(10)
assert paths_for(text_work) == ['draft.txt', 'notes.txt']
paths_for(text_work)

In [ ]:
text_work = catalog.files(root).where(tags.any_of("reviewed", "draft")).where(path.glob("*.txt")).limit(1)
metrics = {
    "displayed": paths_for(text_work),
    "file_count": text_work.count(),
    "content_count": text_work.distinct_content_count(),
    "total_size_bytes": text_work.total_size_bytes(),
}
assert metrics == {
    "displayed": ["draft.txt"],
    "file_count": 2,
    "content_count": 2,
    "total_size_bytes": 10,
}
metrics

## 5. Queries are lazy and observe active tag state

Constructing a query performs no database read. Reusing it after an assertion or retraction observes the then-current active tag state. Retraction preserves assertion history but removes the assertion from current matching.


In [ ]:
temporary_query = catalog.files(root).where(tags.any_of("temporary"))
before = paths_for(temporary_query)
catalog.add_content_tag(notes_id, "temporary", user)
during = paths_for(temporary_query)
catalog.retract_content_tag(notes_id, "temporary", user)
after = paths_for(temporary_query)

assert (before, during, after) == ([], ["notes.txt"], [])
{"before assertion": before, "while active": during, "after retraction": after}

## Exercise

Build a one-row preview of files whose content has either `document` or `empty` from a system producer. Before running the answer, predict the first displayed path, complete file count, distinct-content count, and logical byte total.


In [ ]:
exercise = catalog.files(root).where(tags.any_of("document", "empty", provenance="system")).limit(1)
answer = {
    "displayed": paths_for(exercise),
    "file_count": exercise.count(),
    "content_count": exercise.distinct_content_count(),
    "total_size_bytes": exercise.total_size_bytes(),
}
assert answer == {
    "displayed": ["draft.txt"],
    "file_count": 4,
    "content_count": 3,
    "total_size_bytes": 10,
}
answer

## Pitfalls, extensions, and cleanup

- `len(query.fetch())` is a display count; use the complete reductions for totals.
- Tags describe content identity, not one pathname. Every current copy of those bytes matches.
- `all_of()` does not mean one assertion must supply every tag; each requested tag needs an independently matching active assertion.
- Use `content.is_(group.content_id)` with `stream_all()` to expand one selected duplicate group. Duplicate-group traversal itself remains bounded.

**Extension:** add a user `document` assertion to `notes_id` and predict how the provenance-specific query changes.


In [ ]:
source_after = {path: path.read_bytes() for path in root.rglob("*") if path.is_file()}
assert source_after == source_before
catalog.close()
temporary_directory.cleanup()
"cleanup complete"